In [1]:
%load_ext rpy2.ipython

In [14]:
import pandas as pd
import numpy as np
from Bio import SeqIO
import re

In [3]:
%R pacman::p_load(tidyr, readr, stringr, dplyr, biomaRt)

1,1,1,1,1


In [7]:
%%R
ensembl <- useMart('ensembl')
hsp <- useDataset('hsapiens_gene_ensembl', ensembl)
hsp.gene <- getBM(
	attributes = c(
		'ensembl_gene_id'
	),
	mart = hsp
)
# obtaining orthologs of human
hsp.mus <- getBM(
	attributes = c(
		'external_gene_name',
		'ensembl_gene_id',
		'mmusculus_homolog_ensembl_gene',
		'mmusculus_homolog_orthology_type'
	),
	mart = hsp,
	filters = 'with_mmusculus_homolog',
	values = TRUE
)
hsp.cat <- getBM(
	attributes = c(
		'external_gene_name',
		'ensembl_gene_id',
		'fcatus_homolog_ensembl_gene',
		'fcatus_homolog_orthology_type'
	), 
	mart = hsp,
	filters = 'with_fcatus_homolog',
	values = TRUE
)
hsp.horse <- getBM(
	attributes = c(
		'external_gene_name',
		'ensembl_gene_id',
		'ecaballus_homolog_ensembl_gene',
		'ecaballus_homolog_orthology_type'
	), 
	mart = hsp,
	filters = 'with_ecaballus_homolog',
	values = TRUE
)
hsp.rfe <- getBM(
	attributes = c(
		'external_gene_name',
		'ensembl_gene_id',
		'rferrumequinum_homolog_ensembl_gene',
		'rferrumequinum_homolog_orthology_type'
	),
	mart = hsp,
	filters = 'with_rferrumequinum_homolog',
	values = TRUE
)
hsp.pva <- getBM(
	attributes = c(
		'external_gene_name',
		'ensembl_gene_id',
		'pvampyrus_homolog_ensembl_gene',
		'pvampyrus_homolog_orthology_type'
	),
	mart = hsp,
	filters = 'with_pvampyrus_homolog',
	values = TRUE
)
hsp.mlu <- getBM(
	attributes = c(
		'external_gene_name',
		'ensembl_gene_id',
		'mlucifugus_homolog_ensembl_gene',
		'mlucifugus_homolog_orthology_type'
	),
	mart = hsp,
	filters = 'with_mlucifugus_homolog',
	values = TRUE
)
# filtering and obtaining single copy genes of human
hsp.mus <- as_tibble(hsp.mus) %>% filter(mmusculus_homolog_orthology_type == 'ortholog_one2one')
hsp.cat <- as_tibble(hsp.cat) %>% filter(fcatus_homolog_orthology_type == 'ortholog_one2one')
hsp.horse <- as_tibble(hsp.horse) %>% filter(ecaballus_homolog_orthology_type == 'ortholog_one2one')
hsp.rfe <- as_tibble(hsp.rfe) %>% filter(rferrumequinum_homolog_orthology_type == 'ortholog_one2one')
hsp.pva <- as_tibble(hsp.pva) %>% filter(pvampyrus_homolog_orthology_type == 'ortholog_one2one')
hsp.mlu <- as_tibble(hsp.mlu) %>% filter(mlucifugus_homolog_orthology_type == 'ortholog_one2one')
# mergeing these single copy gene sets together
hsp.single <- hsp.mus %>% inner_join(hsp.cat, by = c('external_gene_name', 'ensembl_gene_id')) %>% inner_join(hsp.horse, by = c('external_gene_name', 'ensembl_gene_id')) %>% inner_join(hsp.rfe, by = c('external_gene_name', 'ensembl_gene_id')) %>% inner_join(hsp.pva, by = c('external_gene_name', 'ensembl_gene_id')) %>% inner_join(hsp.mlu, by = c('external_gene_name', 'ensembl_gene_id')) %>% dplyr::select(external_gene_name, ensembl_gene_id, contains('_homolog_ensembl_gene'))

In [8]:
%R save(hsp.single, file = 'single_of_7.RData')

In [58]:
single_of_7 = %R hsp.single 
single_of_7

,external_gene_name,ensembl_gene_id,mmusculus_homolog_ensembl_gene,fcatus_homolog_ensembl_gene,ecaballus_homolog_ensembl_gene,rferrumequinum_homolog_ensembl_gene,pvampyrus_homolog_ensembl_gene,mlucifugus_homolog_ensembl_gene
1,COLEC12,ENSG00000158270,ENSMUSG00000036103,ENSFCAG00000015762,ENSECAG00000018963,ENSRFEG00010017902,ENSPVAG00000004009,ENSMLUG00000004108
2,IMPA2,ENSG00000141401,ENSMUSG00000024525,ENSFCAG00000045768,ENSECAG00000017598,ENSRFEG00010011097,ENSPVAG00000014183,ENSMLUG00000011183
3,RAB31,ENSG00000168461,ENSMUSG00000056515,ENSFCAG00000044912,ENSECAG00000006886,ENSRFEG00010016775,ENSPVAG00000016514,ENSMLUG00000013590
4,MED15,ENSG00000099917,ENSMUSG00000012114,ENSFCAG00000005999,ENSECAG00000016841,ENSRFEG00010020036,ENSPVAG00000011614,ENSMLUG00000028697
5,YES1,ENSG00000176105,ENSMUSG00000014932,ENSFCAG00000009235,ENSECAG00000021666,ENSRFEG00010017283,ENSPVAG00000000705,ENSMLUG00000023608
...,...,...,...,...,...,...,...,...
10940,IFFO2,ENSG00000169991,ENSMUSG00000041025,ENSFCAG00000044979,ENSECAG00000011016,ENSRFEG00010016288,ENSPVAG00000011748,ENSMLUG00000006129
10941,NTRK1,ENSG00000198400,ENSMUSG00000028072,ENSFCAG00000006913,ENSECAG00000009460,ENSRFEG00010013321,ENSPVAG00000007517,ENSMLUG00000010479
10942,ACOT7,ENSG00000097021,ENSMUSG00000028937,ENSFCAG00000006323,ENSECAG00000023564,ENSRFEG00010009465,ENSPVAG00000013009,ENSMLUG00000003983
10943,CACNA1E,ENSG00000198216,ENSMUSG00000004110,ENSFCAG00000015444,ENSECAG00000012027,ENSRFEG00010020044,ENSPVAG00000004068,ENSMLUG00000017464


Downloading the pep sequences and gff3 files from ensembl to obtain the longest protein sequence of each genes

In [2]:
%%bash
#cd /tmp/ensembl_download
axel -n 10 -a https://ftp.ensembl.org/pub/release-109/fasta/homo_sapiens/pep/Homo_sapiens.GRCh38.pep.all.fa.gz
axel -n 10 -a https://ftp.ensembl.org/pub/release-109/gff3/homo_sapiens/Homo_sapiens.GRCh38.109.chr.gff3.gz

axel -n 10 -a https://ftp.ensembl.org/pub/release-109/gff3/felis_catus/Felis_catus.Felis_catus_9.0.109.chr.gff3.gz
axel -n 10 -a https://ftp.ensembl.org/pub/release-109/fasta/felis_catus/pep/Felis_catus.Felis_catus_9.0.pep.all.fa.gz

axel -n 10 -a https://ftp.ensembl.org/pub/release-109/gff3/mus_musculus/Mus_musculus.GRCm39.109.chr.gff3.gz
axel -n 10 -a https://ftp.ensembl.org/pub/release-109/fasta/mus_musculus/pep/Mus_musculus.GRCm39.pep.all.fa.gz

axel -n 10 -a https://ftp.ensembl.org/pub/release-109/gff3/equus_caballus/Equus_caballus.EquCab3.0.109.chr.gff3.gz
axel -n 10 -a https://ftp.ensembl.org/pub/release-109/fasta/equus_caballus/pep/Equus_caballus.EquCab3.0.pep.all.fa.gz

axel -n 10 -a https://ftp.ensembl.org/pub/release-109/gff3/rhinolophus_ferrumequinum/Rhinolophus_ferrumequinum.mRhiFer1_v1.p.109.chr.gff3.gz
axel -n 10 -a https://ftp.ensembl.org/pub/release-109/fasta/vombatus_ursinus/pep/Vombatus_ursinus.bare-nosed_wombat_genome_assembly.pep.all.fa.gz

axel -n 10 -a https://ftp.ensembl.org/pub/release-109/gff3/pteropus_vampyrus/Pteropus_vampyrus.pteVam1.109.gff3.gz
axel -n 10 -a https://ftp.ensembl.org/pub/release-109/fasta/pteropus_vampyrus/pep/Pteropus_vampyrus.pteVam1.pep.all.fa.gz

axel -n 10 -a https://ftp.ensembl.org/pub/release-109/gff3/myotis_lucifugus/Myotis_lucifugus.Myoluc2.0.109.gff3.gz
axel -n 10 -a https://ftp.ensembl.org/pub/release-109/fasta/myotis_lucifugus/pep/Myotis_lucifugus.Myoluc2.0.pep.all.fa.gz

Initializing download: https://ftp.ensembl.org/pub/release-109/fasta/homo_sapiens/pep/Homo_sapiens.GRCh38.pep.all.fa.gz
File size: 14577980 bytes
Opening output file Homo_sapiens.GRCh38.pep.all.fa.gz
Starting download

Error while terminating subprocess (pid=3251739): 


In [61]:
# the function was used to obtained the longest proteins from gff3 and protein files
def obtain_longest_ensembl(gff3, protein, rna):
    gff_content = open(gff3, 'r')
    gene_lst = {'Name':[], 'ID':[], 'Parent':[]}
    mrna_lst = {'Parent':[], 'mRNA':[]}
    cds_lst = {'Parent':[], 'CDS':[]}
    for line in gff_content:
        if not line.startswith('#'):
            line = line.strip()
            array = line.split('\t')
            feature = array[2]
            attributes = array[8]
            if (feature == 'gene') and ('biotype=protein_coding' in attributes):
                name = attributes.split(';')[1].replace('Name=', '')
                parent = attributes.split(';')[0].replace('ID=gene:', '')
                id = attributes.split(';')[0].replace('ID=gene:', '')
                gene_lst['Name'].append(name)
                gene_lst['ID'].append(id)
                gene_lst['Parent'].append(parent)
            elif feature == 'mRNA':
                parent = attributes.split(';')[1].replace('Parent=gene:', '')
                mrna = attributes.split(';')[0].replace('ID=transcript:', '')
                mrna_lst['Parent'].append(parent)
                mrna_lst['mRNA'].append(mrna)
            elif feature == 'CDS':
                parent = attributes.split(';')[1].replace('Parent=transcript:', '')
                cds = attributes.split(';')[0].replace('ID=CDS:', '')
                cds_lst['Parent'].append(parent)
                cds_lst['CDS'].append(cds)
    gff_content.close()
    gene_tab = pd.DataFrame(gene_lst)
    mrna_tab = pd.DataFrame(mrna_lst)
    cds_tab = pd.DataFrame(cds_lst)
    #print(gene_tab)
    #print(mrna_tab)
    #print(cds_tab)
    
    seq_lst = {'Parent':[], 'SEQ':[], 'Length':[]}
    for record in SeqIO.parse(protein, 'fasta'):
        seq_lst['Parent'].append(record.id.split('.')[0])
        seq_lst['SEQ'].append(str(record.seq))
        seq_lst['Length'].append(len(str(record.seq)))
    seq_tab = pd.DataFrame(seq_lst)
    
    seqrna_lst = {'mRNA':[], 'SEQ':[]}
    for record in SeqIO.parse(rna, 'fasta'):
        seqrna_lst['mRNA'].append(record.id.split('.')[0])
        seqrna_lst['SEQ'].append(str(record.seq))
    seqrna_tab = pd.DataFrame(seqrna_lst)
        
    
    final_tab = pd.merge(gene_tab, mrna_tab, on = 'Parent', how = 'left')
    final_tab = pd.merge(final_tab, cds_tab, left_on='mRNA', right_on='Parent')
    final_tab = pd.merge(final_tab, seq_tab, left_on='CDS', right_on='Parent')
    final_tab = pd.merge(final_tab, seqrna_tab, on = 'mRNA', how='inner', suffixes=['_protein', '_mrna'])
    final_tab = final_tab.drop(columns=['Parent_x', 'Parent_y', 'Parent']).drop_duplicates()
    maxindex = final_tab.groupby(by = 'ID').Length.idxmax()
    final_tab = final_tab.loc[maxindex] 
    return(final_tab)
        

In [54]:
# some protein coding gene is not in gff3 files. Therefore, we extract the longest pep and corresponding cdna from *.cds.all.fa and *.pep.all.fa from ENSEMBL
def obtain_longest_ensembl2(pep, cdna):
    pep_records = SeqIO.parse(pep, 'fasta')
    cdna_records = SeqIO.parse(cdna, 'fasta')
    pep_lst = {'GENE':[], 'PEP':[], 'cDNA':[],'SEQ_pep':[], 'LENGTH':[]}
    cdna_lst = {'cDNA':[], 'SEQ_cdna':[]}
    for record in pep_records:
        descrtiption = record.description
        gene_type = re.search('gene_biotype:[a-zA-Z_]*', descrtiption).group()
        if gene_type == 'gene_biotype:protein_coding':
            pep_lst['PEP'].append(record.id)
            pep_lst['SEQ_pep'].append(str(record.seq))
            pep_lst['LENGTH'].append(len(str(record.seq)))
            gene_id = re.search('gene:[A-Z0-9.]*', descrtiption).group().replace('gene:', '')
            cdna_id = re.search('transcript:[A-Z0-9.]*', descrtiption).group().replace('transcript:', '')
            pep_lst['GENE'].append(gene_id)
            pep_lst['cDNA'].append(cdna_id)
    
    for record in cdna_records:
        cdna_lst['cDNA'].append(record.id)
        cdna_lst['SEQ_cdna'].append(str(record.seq))
    
    pep_tab = pd.DataFrame(pep_lst)
    cdna_tab = pd.DataFrame(cdna_lst)
    final_tab = pd.merge(pep_tab, cdna_tab, how='inner', on = 'cDNA')
    maxindex = final_tab.groupby(by = 'GENE').LENGTH.idxmax()
    final_tab = final_tab.loc[maxindex]
    final_tab.GENE = [i[0] for i in final_tab.GENE.str.split('.')]
    final_tab.PEP = [i[0] for i in final_tab.PEP.str.split('.')]
    final_tab.cDNA = [i[0] for i in final_tab.cDNA.str.split('.')]
    return(final_tab)
        
        
    

Extracting the longest protein sequences of each genes, then these the longest sequences were be intersected with single copy genes to obtain corresponding protein sequences

In [85]:
gff3 = '/tmp/ensembl_download/Homo_sapiens.GRCh38.109.chr.gff3'
protein = '/tmp/ensembl_download/Homo_sapiens.GRCh38.pep.all.fa'
rna = '/tmp/ensembl_download/Homo_sapiens.GRCh38.cds.all.fa'
final_tab = obtain_longest_ensembl2(protein, rna)
# some single copy gene from biomart are not protein coding genes, these genes were discarded by using 'inner' merge
hsp_single = pd.merge(single_of_7.iloc[:, [0, 1]], final_tab, how='inner', left_on='ensembl_gene_id', right_on='GENE')
hsp_single

,external_gene_name,ensembl_gene_id,GENE,PEP,cDNA,SEQ_pep,LENGTH,SEQ_cdna
0,COLEC12,ENSG00000158270,ENSG00000158270,ENSP00000383115,ENST00000400256,MKDDFAEEEEVQSFGYKRFGIQEGTQCTKCKNNWALKFSIILLYIL...,742,ATGAAAGACGACTTCGCAGAGGAGGAGGAGGTGCAATCCTTCGGTT...
1,IMPA2,ENSG00000141401,ENSG00000141401,ENSP00000269159,ENST00000269159,MKPSGEDQAALAAGPWEECFQAAVQLALRAGQIIRKALTEEKRVST...,288,ATGAAGCCGAGCGGCGAGGACCAGGCGGCGCTGGCGGCCGGCCCCT...
2,RAB31,ENSG00000168461,ENSG00000168461,ENSP00000461945,ENST00000578921,MMAIRELKVCLLGDTGVGKSSIVCRFVQDHFDHNISPTIGASFMTK...,195,ATGATGGCGATACGGGAGCTCAAAGTGTGCCTTCTCGGGGACACTG...
3,MED15,ENSG00000099917,ENSG00000099917,ENSP00000263205,ENST00000263205,MDVSGQETDWRSTAFRQKLVSQIEDAMRKAGVAHSKSSKDMESHVF...,788,ATGGACGTTTCCGGGCAAGAGACCGACTGGCGGAGCACCGCCTTCC...
4,YES1,ENSG00000176105,ENSG00000176105,ENSP00000464380,ENST00000577961,MLDLIMGCIKSKENKSPAIKYRPENTPEPVSTSVSHYGAEPTTVSP...,548,ATGTTAGATTTGATAATGGGCTGCATTAAAAGTAAAGAAAACAAAA...
...,...,...,...,...,...,...,...,...
10874,IFFO2,ENSG00000169991,ENSG00000169991,ENSP00000387941,ENST00000455833,MVNSLLFGEMALAFGCPPGGGGGGCPGGGGGGGGAGPGPSPVTAAL...,517,ATGGTGAACTCGCTGCTGTTCGGGGAGATGGCCTTGGCCTTCGGCT...
10875,NTRK1,ENSG00000198400,ENSG00000198400,ENSP00000431418,ENST00000524377,MLRGGRRGQLGWHSWAAGPGSLLAWLILASAGAAPCPDACCPHGSS...,796,ATGCTGCGAGGCGGACGGCGCGGGCAGCTTGGCTGGCACAGCTGGG...
10876,ACOT7,ENSG00000097021,ENSG00000097021,ENSP00000367086,ENST00000377855,MKLLARALRLCEFGRQASSRRLVAGQGCVGPRRGCCAPVQVVGPRA...,380,ATGAAGCTGCTTGCCAGGGCTCTCCGGCTCTGTGAGTTTGGGAGGC...
10877,CACNA1E,ENSG00000198216,ENSG00000198216,ENSP00000356545,ENST00000367573,MARFGEAVVARPGSGDGDSDQSRNRQGTPVPASGQAAAYKQTKAQR...,2313,ATGGCTCGCTTCGGGGAGGCGGTGGTCGCCAGGCCAGGGTCCGGCG...


In [87]:
gff3 = '/tmp/ensembl_download/Mus_musculus.GRCm39.109.chr.gff3'
protein = '/tmp/ensembl_download/Mus_musculus.GRCm39.pep.all.fa'
rna = '/tmp/ensembl_download/Mus_musculus.GRCm39.cds.all.fa'
final_tab = obtain_longest_ensembl2(protein, rna)
mus_signle = pd.merge(single_of_7.iloc[:, [0, 1, 2]], final_tab, how='inner', left_on='mmusculus_homolog_ensembl_gene', right_on='GENE')
mus_signle

,external_gene_name,ensembl_gene_id,mmusculus_homolog_ensembl_gene,GENE,PEP,cDNA,SEQ_pep,LENGTH,SEQ_cdna
0,COLEC12,ENSG00000158270,ENSMUSG00000036103,ENSMUSG00000036103,ENSMUSP00000157132,ENSMUST00000234965,MKDDFAEEEEVQSFGYKRFGIQEGTQCTKCKNNWALKFSIVLLYIL...,742,ATGAAAGACGACTTTGCAGAGGAAGAGGAGGTGCAGTCCTTCGGTT...
1,IMPA2,ENSG00000141401,ENSMUSG00000024525,ENSMUSG00000024525,ENSMUSP00000025403,ENSMUST00000025403,MKPSSEEEGELVQGVGPWDECFEVAVQLALRAGQIIRKALTEEKRV...,290,ATGAAGCCGAGCAGCGAGGAAGAGGGAGAGTTGGTGCAGGGCGTGG...
2,RAB31,ENSG00000168461,ENSMUSG00000056515,ENSMUSG00000056515,ENSMUSP00000068195,ENSMUST00000070673,MMAIRELKVCLLGDTGVGKSSIVCRFVQDHFDHNISPTIGASFMTK...,195,ATGATGGCGATACGGGAGCTCAAAGTGTGTCTTCTCGGGGACACGG...
3,MED15,ENSG00000099917,ENSMUSG00000012114,ENSMUSG00000012114,ENSMUSP00000012259,ENSMUST00000012259,MDVSGQETDWRSAAFRQKLVSQIEDAMRKAGVAHSKSSKDMESHVF...,789,ATGGACGTTTCGGGGCAGGAGACCGACTGGCGTAGCGCCGCCTTTC...
4,YES1,ENSG00000176105,ENSMUSG00000014932,ENSMUSG00000014932,ENSMUSP00000144001,ENSMUST00000202543,MGCIKSKENKSPAIKYTPENLTEPVSPSASHYGVEHATVAPTSSTK...,541,ATGGGCTGCATTAAAAGTAAAGAAAACAAAAGTCCAGCCATAAAAT...
...,...,...,...,...,...,...,...,...,...
10874,IFFO2,ENSG00000169991,ENSMUSG00000041025,ENSMUSG00000041025,ENSMUSP00000134062,ENSMUST00000174078,MVNSLLFGEMALAFGCPPGGGGCAGGGGGGGAGPGPSPVTAALRDD...,512,ATGGTTAACTCGCTGCTGTTCGGAGAGATGGCCTTGGCCTTCGGCT...
10875,NTRK1,ENSG00000198400,ENSMUSG00000028072,ENSMUSG00000028072,ENSMUSP00000029712,ENSMUST00000029712,MLRGQRLGQLGWHRPAAGLGSLMTSLMLACASAASCREVCCPVGPS...,799,ATGCTGCGAGGCCAGCGGCTCGGGCAGCTGGGCTGGCATCGCCCGG...
10876,ACOT7,ENSG00000097021,ENSMUSG00000028937,ENSMUSG00000028937,ENSMUSP00000129121,ENSMUST00000167926,MSAMKLLVGTLRLWEVGRQVAFSSLTPGQECSGLRKTFWAAMRAVR...,384,ATGTCTGCAATGAAGCTGCTGGTCGGGACTCTGCGCCTCTGGGAGG...
10877,CACNA1E,ENSG00000198216,ENSMUSG00000004110,ENSMUSG00000004110,ENSMUSP00000140937,ENSMUST00000187541,MARFGEAVVVGRPGSGDGDSDQSRNRQGTPVPASGPAAAYKQSKAQ...,2273,ATGGCTCGCTTCGGGGAGGCGGTGGTCGTTGGCAGGCCAGGCTCAG...


In [88]:
gff3 = '/tmp/ensembl_download/Felis_catus.Felis_catus_9.0.109.chr.gff3'
protein = '/tmp/ensembl_download/Felis_catus.Felis_catus_9.0.pep.all.fa'
rna = '/tmp/ensembl_download/Felis_catus.Felis_catus_9.0.cds.all.fa'
final_tab = obtain_longest_ensembl2(protein, rna)
cat_signle = pd.merge(single_of_7.iloc[:, [0, 1, 3]], final_tab, how='inner', left_on='fcatus_homolog_ensembl_gene', right_on='GENE')
cat_signle

,external_gene_name,ensembl_gene_id,fcatus_homolog_ensembl_gene,GENE,PEP,cDNA,SEQ_pep,LENGTH,SEQ_cdna
0,COLEC12,ENSG00000158270,ENSFCAG00000015762,ENSFCAG00000015762,ENSFCAP00000026359,ENSFCAT00000037896,MKDDFAEEEEVQSFGYKRFGIQEGTQCTKCKNNWALKFSIILLYIL...,753,ATGAAAGACGACTTTGCAGAGGAGGAGGAGGTGCAGTCCTTCGGTT...
1,IMPA2,ENSG00000141401,ENSFCAG00000045768,ENSFCAG00000045768,ENSFCAP00000059502,ENSFCAT00000076264,MKPNGEDEEAPAGGPWEECFEAAVQLALRAGQIIRKALSEEKRVST...,560,ATGAAGCCGAACGGCGAGGACGAGGAGGCGCCCGCCGGGGGCCCCT...
2,RAB31,ENSG00000168461,ENSFCAG00000044912,ENSFCAG00000044912,ENSFCAP00000053764,ENSFCAT00000069114,MSFISSSSGTLLGRNGHGKYRPGSKWRRLGADRTQDPIRQHTESAP...,210,ATGAGCTTCATAAGTTCCTCATCTGGGACACTGCTGGGCAGGAACG...
3,MED15,ENSG00000099917,ENSFCAG00000005999,ENSFCAG00000005999,ENSFCAP00000056989,ENSFCAT00000080064,MDVSGQETDWRSAAFRQKLVSQIEDAMKAGVAHSKSSKDMESHVFL...,788,ATGGACGTTTCGGGGCAGGAGACCGACTGGCGGAGCGCCGCCTTCC...
4,YES1,ENSG00000176105,ENSFCAG00000009235,ENSFCAG00000009235,ENSFCAP00000048850,ENSFCAT00000055295,VVDLIMGCIKSKENKSPTIKYRTESTPEPVSASVSHYGAEHTAVAP...,546,GTTGTAGATTTGATAATGGGCTGCATTAAAAGTAAAGAGAACAAAA...
...,...,...,...,...,...,...,...,...,...
10874,IFFO2,ENSG00000169991,ENSFCAG00000044979,ENSFCAG00000044979,ENSFCAP00000044791,ENSFCAT00000053996,MVNSLLFGEMALAFGCPPGGGGGSCPGGGGGGGGAGPGPSPVTAAL...,517,ATGGTGAACTCGCTGCTGTTCGGGGAGATGGCCTTGGCCTTCGGCT...
10875,NTRK1,ENSG00000198400,ENSFCAG00000006913,ENSFCAG00000006913,ENSFCAP00000046864,ENSFCAT00000057198,MLASAGAAPCADVCCPHGPSGLRCTRAGALESLRRLPGAENLTELY...,770,ATGCTGGCGTCCGCGGGCGCCGCACCCTGCGCGGACGTCTGCTGCC...
10876,ACOT7,ENSG00000097021,ENSFCAG00000006323,ENSFCAG00000006323,ENSFCAP00000005876,ENSFCAT00000006325,MAPPGLIHSAPGLPDTCVLFQSFADAASMSGPAAETPSAIQICRIM...,445,ATGGCGCCGCCCGGGCTCATTCATTCCGCGCCAGGCCTGCCAGACA...
10877,CACNA1E,ENSG00000198216,ENSFCAG00000015444,ENSFCAG00000015444,ENSFCAP00000014321,ENSFCAT00000015448,MVPEMRGPSPGHRLPEPAPCPAAGGNPEPWALGRRHRHVLALPEEE...,2455,ATGGTGCCGGAGATGCGTGGACCGAGCCCAGGCCACAGGCTGCCCG...


In [89]:
gff3 = '/tmp/ensembl_download/Equus_caballus.EquCab3.0.109.chr.gff3'
protein = '/tmp/ensembl_download/Equus_caballus.EquCab3.0.pep.all.fa'
rna = '/tmp/ensembl_download/Equus_caballus.EquCab3.0.cds.all.fa'
final_tab = obtain_longest_ensembl2(protein, rna)
horse_signle = pd.merge(single_of_7.iloc[:, [0, 1, 4]], final_tab, how='inner', left_on='ecaballus_homolog_ensembl_gene', right_on='GENE')
horse_signle

,external_gene_name,ensembl_gene_id,ecaballus_homolog_ensembl_gene,GENE,PEP,cDNA,SEQ_pep,LENGTH,SEQ_cdna
0,COLEC12,ENSG00000158270,ENSECAG00000018963,ENSECAG00000018963,ENSECAP00000032296,ENSECAT00000062145,MLEKRGDKENSGPLGRFLQNSTCATVVFTWGMLWRWREVEKCEIYF...,807,ATGTTGGAGAAAAGAGGGGATAAGGAGAACAGTGGACCACTTGGAC...
1,IMPA2,ENSG00000141401,ENSECAG00000017598,ENSECAG00000017598,ENSECAP00000015480,ENSECAT00000018944,MKLVAAGQTFLLEHCLSCPPSLQAAHLLPRIIRKALSEEKRVSTKT...,286,ATGAAACTTGTAGCTGCAGGGCAAACATTCTTGCTGGAGCACTGTC...
2,RAB31,ENSG00000168461,ENSECAG00000006886,ENSECAG00000006886,ENSECAP00000047871,ENSECAT00000068518,MLQPEAISLIRESGVLQIMRPYFQMEEETQDKSPGLRLNLPGSELG...,217,ATGCTGCAGCCTGAGGCCATTTCCTTGATCCGAGAGAGCGGTGTTC...
3,MED15,ENSG00000099917,ENSECAG00000016841,ENSECAG00000016841,ENSECAP00000014878,ENSECAT00000018251,MDFVRPAHSRVTGNAGGQAPACCRQHREDYAATAPPRPLSSARGIW...,873,ATGGACTTTGTGAGGCCAGCACACAGTAGGGTCACAGGAAATGCTG...
4,YES1,ENSG00000176105,ENSECAG00000021666,ENSECAG00000021666,ENSECAP00000058347,ENSECAT00000138207,MPCDNRGRDGSDAVASQGTPRIASHNQKLGGGKEGFSSEIENLIMG...,585,ATGCCGTGTGACAACAGAGGCAGAGATGGAAGTGATGCAGTTGCGA...
...,...,...,...,...,...,...,...,...,...
10874,IFFO2,ENSG00000169991,ENSECAG00000011016,ENSECAG00000011016,ENSECAP00000008931,ENSECAT00000011425,MLAPLRGRAGSAAVPANRVPGAALRSASPGEESEPGRAAGPGAGDG...,752,ATGCTGGCCCCTTTAAGAGGCCGGGCCGGCTCTGCGGCGGTTCCAG...
10875,NTRK1,ENSG00000198400,ENSECAG00000009460,ENSECAG00000009460,ENSECAP00000007660,ENSECAT00000009976,MSREARQPLLRAHRRRPGRGEAGAAAMLRGGRRGQLGWHGRATGPG...,822,ATGTCGCGGGAGGCCCGGCAGCCGCTGCTGCGAGCGCACAGACGGC...
10876,ACOT7,ENSG00000097021,ENSECAG00000023564,ENSECAG00000023564,ENSECAP00000077311,ENSECAT00000092944,MNERSFTGNRPRFPLAADFRSRDPGGDSRRPLCRTPATAHRGRAGR...,495,ATGAATGAACGCTCGTTTACGGGAAACCGCCCCCGCTTTCCCCTCG...
10877,CACNA1E,ENSG00000198216,ENSECAG00000012027,ENSECAG00000012027,ENSECAP00000010876,ENSECAT00000013682,MARFGEAVVGRPGSGDGDSDQSRNRQGTPVPASGPAAAYKQSKAQR...,2315,ATGGCTCGCTTCGGGGAGGCGGTGGTCGGCAGGCCCGGGTCAGGCG...


In [90]:
gff3 = '/tmp/ensembl_download/Rhinolophus_ferrumequinum.mRhiFer1_v1.p.109.chr.gff3'
protein = '/tmp/ensembl_download/Rhinolophus_ferrumequinum.mRhiFer1_v1.p.pep.all.fa'
rna = '/tmp/ensembl_download/Rhinolophus_ferrumequinum.mRhiFer1_v1.p.cds.all.fa'
final_tab = obtain_longest_ensembl2(protein, rna)
rfe_single = pd.merge(single_of_7.iloc[:, [0, 1, 5]], final_tab, how='inner', left_on='rferrumequinum_homolog_ensembl_gene', right_on='GENE')
rfe_single

,external_gene_name,ensembl_gene_id,rferrumequinum_homolog_ensembl_gene,GENE,PEP,cDNA,SEQ_pep,LENGTH,SEQ_cdna
0,COLEC12,ENSG00000158270,ENSRFEG00010017902,ENSRFEG00010017902,ENSRFEP00010026991,ENSRFET00010029328,MIRGTMVYWRDRGFCSQTDPGIQEGTQCTKCKNNWALKFSIILLYI...,743,ATGATCAGAGGCACCATGGTGTACTGGAGAGATCGGGGCTTTTGCA...
1,IMPA2,ENSG00000141401,ENSRFEG00010011097,ENSRFEG00010011097,ENSRFEP00010016367,ENSRFET00010017871,MKPGREDEAAPAGGPWEECFEVAVQLALRAGQIIRKALSEEKRVST...,288,ATGAAGCCAGGTCGCGAGGACGAGGCGGCGCCCGCCGGGGGCCCCT...
2,RAB31,ENSG00000168461,ENSRFEG00010016775,ENSRFEG00010016775,ENSRFEP00010025232,ENSRFET00010027422,MLSPYPDRPRAHDGDTGAQSVSSRGWPWLTGSCSLMKCQCWQESGK...,247,ATGCTGAGCCCCTACCCGGACCGACCCCGAGCACATGATGGCGATA...
3,MED15,ENSG00000099917,ENSRFEG00010020036,ENSRFEG00010020036,ENSRFEP00010030406,ENSRFET00010032975,MDVSGQETDWRSAAFRQKLVSQIEDAMRKAGVAHSKSSKDMESHVF...,793,ATGGACGTTTCGGGGCAGGAGACCGATTGGCGGAGCGCCGCCTTCC...
4,YES1,ENSG00000176105,ENSRFEG00010017283,ENSRFEG00010017283,ENSRFEP00010026216,ENSRFET00010028484,VDLIMGCIKSKENKSPAIKYTSENTPEPVSTSVSHYGAEHTAATPS...,545,GTAGATTTGATAATGGGCTGTATTAAAAGTAAAGAGAACAAAAGTC...
...,...,...,...,...,...,...,...,...,...
10874,IFFO2,ENSG00000169991,ENSRFEG00010016288,ENSRFEG00010016288,ENSRFEP00010024614,ENSRFET00010026752,MVNSLLFGEMALAFGCPPGGGGGGCPGGGGGGVGPGPSPVTAALRD...,515,ATGGTGAACTCGCTGCTGTTCGGAGAGATGGCCTTGGCCTTCGGCT...
10875,NTRK1,ENSG00000198400,ENSRFEG00010013321,ENSRFEG00010013321,ENSRFEP00010021293,ENSRFET00010023172,MLRGGGRRQLGRHGRPTGPGSLLAWLMLASAGAAPCSDVCCPHGPS...,796,ATGCTGCGAGGCGGTGGGCGCCGGCAGCTTGGCCGGCACGGCCGGC...
10876,ACOT7,ENSG00000097021,ENSRFEG00010009465,ENSRFEG00010009465,ENSRFEP00010014235,ENSRFET00010015562,MAPPPPSPSMAPPGLIHSAPGPPDTCALLQLPAAAAASMSGPTTET...,376,ATGGCCCCGCCTCCTCCCTCCCCCTCCATGGCGCCGCCCGGGCTCA...
10877,CACNA1E,ENSG00000198216,ENSRFEG00010020044,ENSRFEG00010020044,ENSRFEP00010031427,ENSRFET00010034074,MARFGEAVVGRPGSGDGDSDQSRNRQGTPVPASGPAAAYKQSKAQR...,2314,ATGGCTCGCTTCGGGGAGGCGGTGGTCGGCAGGCCGGGGTCGGGTG...


In [91]:
gff3 = '/tmp/ensembl_download/Pteropus_vampyrus.pteVam1.109.gff3'
protein = '/tmp/ensembl_download/Pteropus_vampyrus.pteVam1.pep.all.fa'
rna = '/tmp/ensembl_download/Pteropus_vampyrus.pteVam1.cds.all.fa'
final_tab = obtain_longest_ensembl2(protein, rna)
pva_single = pd.merge(single_of_7.iloc[:, [0, 1, 6]], final_tab, how='inner', left_on='pvampyrus_homolog_ensembl_gene', right_on='GENE')
pva_single

,external_gene_name,ensembl_gene_id,pvampyrus_homolog_ensembl_gene,GENE,PEP,cDNA,SEQ_pep,LENGTH,SEQ_cdna
0,COLEC12,ENSG00000158270,ENSPVAG00000004009,ENSPVAG00000004009,ENSPVAP00000003793,ENSPVAT00000004008,DFAEEEEVQSFGYKRFGIQEGTQCTKCKNNWALKFSIILLYILCAL...,733,GACTTCGCGGAGGAGGAGGAGGTGCAGTCCTTCGGTTATAAGCGGT...
1,IMPA2,ENSG00000141401,ENSPVAG00000014183,ENSPVAG00000014183,ENSPVAP00000013365,ENSPVAT00000014183,MKPSREDQAAPAGGPWEECFEAAVQLALRAGQIIRKALSEEKRVST...,250,ATGAAGCCAAGCCGCGAGGACCAGGCGGCGCCCGCTGGGGGCCCCT...
2,RAB31,ENSG00000168461,ENSPVAG00000016514,ENSPVAG00000016514,ENSPVAP00000015583,ENSPVAT00000016514,YFLSDTGVGKSSIVCRFVQDHFDHNISPTIGASFMTKTVPCGNELH...,186,TATTTTTTGAGTGACACCGGAGTTGGGAAATCGAGCATTGTGTGCC...
3,MED15,ENSG00000099917,ENSPVAG00000011614,ENSPVAG00000011614,ENSPVAP00000010948,ENSPVAT00000011616,MDVSGQETDWRTAAFRQKLVSQIEDAMRKAGVAHSKSSKDMESHVF...,783,ATGGACGTTTCGGGGCAGGAGACCGATTGGCGGACCGCCGCCTTCC...
4,YES1,ENSG00000176105,ENSPVAG00000000705,ENSPVAG00000000705,ENSPVAP00000000667,ENSPVAT00000000705,MGCIKSKENKSPAIKHRTENTTESVNTSVTHYGAEHTAATPTSTAK...,540,ATGGGCTGCATTAAAAGTAAAGAGAACAAAAGTCCAGCCATTAAAC...
...,...,...,...,...,...,...,...,...,...
10874,IFFO2,ENSG00000169991,ENSPVAG00000011748,ENSPVAG00000011748,ENSPVAP00000011076,ENSPVAT00000011748,WSYTQVRRTGGGGVETVQGPGVSWVHPDGVGVQIDTITPEIRALYN...,356,TGGAGCTACACGCAGGTGCGGCGCACTGGCGGCGGCGGCGTGGAGA...
10875,NTRK1,ENSG00000198400,ENSPVAG00000007517,ENSPVAG00000007517,ENSPVAP00000007095,ENSPVAT00000007517,MLRGGRRGQLGGRGGATGPGGLLAWLVLASAGAAPCPDVCCPRGPS...,796,ATGCTGCGAGGCGGACGGCGCGGGCAGCTCGGCGGGCGCGGCGGGG...
10876,ACOT7,ENSG00000097021,ENSPVAG00000013009,ENSPVAG00000013009,ENSPVAP00000012267,ENSPVAT00000013008,IKLPAWALCLRAFAGGCLGPRPPGQLWERLASGSAVATKSEMLSGS...,341,ATAAAGCTGCCGGCATGGGCTCTCTGCCTGCGGGCATTTGCCGGGG...
10877,CACNA1E,ENSG00000198216,ENSPVAG00000004068,ENSPVAG00000004068,ENSPVAP00000003854,ENSPVAT00000004073,MARFGEAVVARPGSGDGDSDQSRNRQGTPVPASGPAAAYKQSKAQR...,2314,ATGGCTCGCTTCGGGGAGGCGGTGGTCGCCAGGCCGGGGTCAGGCG...


In [92]:
gff3 = '/tmp/ensembl_download/Myotis_lucifugus.Myoluc2.0.109.gff3'
protein = '/tmp/ensembl_download/Myotis_lucifugus.Myoluc2.0.pep.all.fa'
rna = '/tmp/ensembl_download/Myotis_lucifugus.Myoluc2.0.cds.all.fa'
final_tab = obtain_longest_ensembl2(protein, rna)
mlu_single = pd.merge(single_of_7.iloc[:, [0, 1, 7]], final_tab, how='inner', left_on='mlucifugus_homolog_ensembl_gene', right_on='GENE')
mlu_single

,external_gene_name,ensembl_gene_id,mlucifugus_homolog_ensembl_gene,GENE,PEP,cDNA,SEQ_pep,LENGTH,SEQ_cdna
0,COLEC12,ENSG00000158270,ENSMLUG00000004108,ENSMLUG00000004108,ENSMLUP00000003742,ENSMLUT00000004107,LSEWNLVQNYSMRQRGIQEGTQCTKCKNNWALKFSIILLYILCALL...,740,TTATCAGAGTGGAATTTGGTTCAAAACTATTCTATGAGACAAAGAG...
1,IMPA2,ENSG00000141401,ENSMLUG00000011183,ENSMLUG00000011183,ENSMLUP00000010182,ENSMLUT00000011171,EVAGPREDAMKPSLEDPAASAGGPWEECFELAVQLALRAGQIIRKA...,297,GAGGTGGCGGGGCCCCGCGAGGACGCGATGAAGCCAAGCCTCGAGG...
2,RAB31,ENSG00000168461,ENSMLUG00000013590,ENSMLUG00000013590,ENSMLUP00000012368,ENSMLUT00000013592,MSEIQKLHTSFLLDTGVGKSSIVCRFVQDHFDHNISPTIGASFMTK...,195,ATGTCAGAAATACAGAAATTACATACTTCTTTTCTACTGGACACCG...
3,MED15,ENSG00000099917,ENSMLUG00000028697,ENSMLUG00000028697,ENSMLUP00000022598,ENSMLUT00000027136,GHQRKTRVSQNSLTMQGQQVQTPQSMPPPPRPSPQPGQPSSQPNSN...,366,GGGCATCAGAGAAAGACAAGGGTCAGCCAGAACAGCCTCACCATGC...
4,YES1,ENSG00000176105,ENSMLUG00000023608,ENSMLUG00000023608,ENSMLUP00000017787,ENSMLUT00000023230,MGCIKSKENKSPAIKYRTENTPEPVSTSVSHYGTEHTPATPSSSAK...,544,ATGGGCTGCATTAAAAGTAAAGAGAATAAAAGTCCAGCAATTAAAT...
...,...,...,...,...,...,...,...,...,...
10874,IFFO2,ENSG00000169991,ENSMLUG00000006129,ENSMLUG00000006129,ENSMLUP00000005597,ENSMLUT00000006130,GPGPSPVTAALRDDLGSNIHLLKGLNVRFRCFLAKVHELERRNRLL...,450,GGGCCAGGGCCATCGCCGGTGACGGCGGCGTTGCGGGATGACCTGG...
10875,NTRK1,ENSG00000198400,ENSMLUG00000010479,ENSMLUG00000010479,ENSMLUP00000009562,ENSMLUT00000010490,YIENEQHLQRLEPNHLRGLGELRNLTIVKSGLRSVAPNAFHFTPRL...,716,TACATCGAGAATGAGCAGCATCTGCAGCGTCTGGAGCCCAATCACC...
10876,ACOT7,ENSG00000097021,ENSMLUG00000003983,ENSMLUG00000003983,ENSMLUP00000018289,ENSMLUT00000003983,WRTLSSPRLSLFLRTSFPEASSCLFPPPRIMRPDDANVAGNVHGGT...,465,TGGAGGACCTTGTCCTCTCCCCGGTTGTCACTGTTCCTGAGGACGT...
10877,CACNA1E,ENSG00000198216,ENSMLUG00000017464,ENSMLUG00000017464,ENSMLUP00000015925,ENSMLUT00000017471,PFEYMILATIIANCIVLALEQHLPEDDKTPMSRRLEKTEPYFIGIF...,2168,CCGTTTGAGTACATGATCCTGGCCACCATCATTGCCAACTGCATCG...


In [83]:
# save these single copy protein sequencs in fasta file. Those fasta files will be used in downstream anlysis
def toFasta(dataframe, output):
    outfile_protein = open(f'/home/panda2bat/Avivorous_bat/input/Download/Protein/ensembl_single_copy/{output}.1to1.faa', 'w')
    outfile_cds = open(f'/home/panda2bat/Avivorous_bat/input/Download/CDS/ensembl_single_copy/{output}.1to1.fna', 'w') 
    for index, data in dataframe.iterrows():
        outfile_protein.write(f">{data.ensembl_gene_id}_{data.external_gene_name}|{output}\n{data.SEQ_pep}\n")
        outfile_cds.write(f">{data.ensembl_gene_id}_{data.external_gene_name}|{output}\n{data.SEQ_cdna}\n")
    outfile_protein.close()
    outfile_cds.close()

In [93]:
toFasta(hsp_single, 'HomSap')
toFasta(mus_signle, 'MusMus')
toFasta(cat_signle, 'FelCat')
toFasta(horse_signle, 'EquCab')
toFasta(rfe_single, 'RhiFer')
toFasta(pva_single, 'PteVam')
toFasta(mlu_single, 'MyoLuc')

In [11]:
single_of_7.query('ensembl_gene_id == "ENSG00000144908"')

,external_gene_name,ensembl_gene_id,mmusculus_homolog_ensembl_gene,fcatus_homolog_ensembl_gene,ecaballus_homolog_ensembl_gene,rferrumequinum_homolog_ensembl_gene,pvampyrus_homolog_ensembl_gene,mlucifugus_homolog_ensembl_gene
5481,ALDH1L1,ENSG00000144908,ENSMUSG00000030088,ENSFCAG00000040881,ENSECAG00000021048,ENSRFEG00010009061,ENSPVAG00000006890,ENSMLUG00000009750


In [9]:
single_of_7

,external_gene_name,ensembl_gene_id,mmusculus_homolog_ensembl_gene,fcatus_homolog_ensembl_gene,ecaballus_homolog_ensembl_gene,rferrumequinum_homolog_ensembl_gene,pvampyrus_homolog_ensembl_gene,mlucifugus_homolog_ensembl_gene
1,COLEC12,ENSG00000158270,ENSMUSG00000036103,ENSFCAG00000015762,ENSECAG00000018963,ENSRFEG00010017902,ENSPVAG00000004009,ENSMLUG00000004108
2,IMPA2,ENSG00000141401,ENSMUSG00000024525,ENSFCAG00000045768,ENSECAG00000017598,ENSRFEG00010011097,ENSPVAG00000014183,ENSMLUG00000011183
3,RAB31,ENSG00000168461,ENSMUSG00000056515,ENSFCAG00000044912,ENSECAG00000006886,ENSRFEG00010016775,ENSPVAG00000016514,ENSMLUG00000013590
4,MED15,ENSG00000099917,ENSMUSG00000012114,ENSFCAG00000005999,ENSECAG00000016841,ENSRFEG00010020036,ENSPVAG00000011614,ENSMLUG00000028697
5,YES1,ENSG00000176105,ENSMUSG00000014932,ENSFCAG00000009235,ENSECAG00000021666,ENSRFEG00010017283,ENSPVAG00000000705,ENSMLUG00000023608
...,...,...,...,...,...,...,...,...
10940,IFFO2,ENSG00000169991,ENSMUSG00000041025,ENSFCAG00000044979,ENSECAG00000011016,ENSRFEG00010016288,ENSPVAG00000011748,ENSMLUG00000006129
10941,NTRK1,ENSG00000198400,ENSMUSG00000028072,ENSFCAG00000006913,ENSECAG00000009460,ENSRFEG00010013321,ENSPVAG00000007517,ENSMLUG00000010479
10942,ACOT7,ENSG00000097021,ENSMUSG00000028937,ENSFCAG00000006323,ENSECAG00000023564,ENSRFEG00010009465,ENSPVAG00000013009,ENSMLUG00000003983
10943,CACNA1E,ENSG00000198216,ENSMUSG00000004110,ENSFCAG00000015444,ENSECAG00000012027,ENSRFEG00010020044,ENSPVAG00000004068,ENSMLUG00000017464


In [86]:
toFasta(hsp_single, 'HomSap')

In [16]:
final_tab.query("ID == 'ENSFCAG00000040881'")

,Name,ID,mRNA,CDS,SEQ_protein,Length,SEQ_mrna
